# Inter-Coder Reliability — Step 2

Sentence-level reliability for the full social-group label inventory.Reliability is reported as Krippendorff's α (Krippendorff 2004) with MASI distance (Passonneau 2006) to handle multi-label agreement.

In [ ]:
import ast, re
from pathlib import Path

import numpy as np
import pandas as pd
import krippendorff

In [ ]:
CONSENSUS_FILE = Path('./inputs/consensus.xlsx')   # consensus column: 'ground_truth'
ANN_FILE       = Path('./inputs/annotations.xlsx') # columns: text, annotator, label
OUTPUT_DIR     = Path('./results')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Codebook

In [ ]:
CODEBOOK: dict[str, list[str]] = {
    'Socio-economic position': [
        'lower class', 'middle class', 'upper class',
        'capital owners, investors and shareholders',
        'unskilled or unqualified', 'skilled or qualified',
    ],
    'Labor market position': [
        'wage and salary earners', 'civil servants', 'CEOs and corporate leaders',
        'employers', 'entrepreneurs', 'self-employed and freelancers',
        'unemployed', 'retirees', 'housewives and househusbands',
    ],
    'Age and family status': [
        'parents and families', 'minors', 'youth',
        'middle-aged and pre-retirement age groups', 'elderly', 'couples', 'singles',
    ],
    'Identities and minority/majority status': [
        'men', 'women', 'cisgender and heterosexuals', 'lgbtqia+', 'disabled people',
        'people with an immigration background, including immigrants',
        'ethnic and racial minorities',
        'christians', 'jews', 'muslims',
        'multiple (or other) religious or minority groups',
    ],
    'Profession': [
        'athletes', 'authors and artists', 'doctors', 'farmers and fishermen',
        'health and care professionals', 'journalists', 'legal professionals',
        'politicians and high-ranking officials', 'sex workers',
        'scientists and professors', 'security forces', 'soldiers',
        'teachers and educators', 'other professions',
    ],
    'Social roles and behavior': [
        'consumers and clients', 'car drivers', 'patients',
    ],
    'Social deviance': [
        'extremists',
        'terrorists, rebels, revolutionaries and/or movements of armed resistance',
        'offenders, criminals, prisoners and/or accused people',
        'drug addicts',
    ],
    'Real estate ownership': ['real-estate owners', 'tenants', 'homeless'],
    'Others': ['others'],
}

CANONICAL = {lbl for lbls in CODEBOOK.values() for lbl in lbls}

## Label normalisation

Legacy and variant label spellings are mapped to canonical codebook labels. Unknown labels fall back to `'others'`.

In [ ]:
COMMA_LABELS = [
    'offenders, criminals, prisoners and/or accused people',
    'terrorists, rebels, revolutionaries and/or movements of armed resistance',
    'capital owners, investors and shareholders',
    'multiple (or other) religious or minority groups',
]

NORMALISE: dict[str, str | None] = {
    'poor': 'lower class', 'underprivileged': 'lower class',
    'unskilled and/or underprivileged': 'unskilled or unqualified',
    'qualified': 'skilled or qualified', 'qualified and graduates': 'skilled or qualified',
    'investors and stakeholders': 'capital owners, investors and shareholders',
    'employees': 'wage and salary earners', 'precarious employees': 'wage and salary earners',
    'working active population': 'wage and salary earners',
    'housewife and househusband': 'housewives and househusbands',
    'self-employed/freelancers': 'self-employed and freelancers',
    'leaders': 'CEOs and corporate leaders',
    'ceos and corporate leaders': 'CEOs and corporate leaders',
    'enterprises': 'entrepreneurs', 'large enterprises': 'entrepreneurs',
    'small- and middle-size enterprises': 'entrepreneurs', 'specific sector': 'entrepreneurs',
    'entrepreneurs (smes)': 'entrepreneurs', 'entrepreneurs (large enterprises)': 'entrepreneurs',
    'entrepreneurs in [specific] sector': 'entrepreneurs',
    'minors, including children and pupils': 'minors',
    'youth, including students and apprentices': 'youth',
    'middle-aged': 'middle-aged and pre-retirement age groups',
    'older age group': 'elderly', 'seniors': 'elderly',
    'immigrants': 'people with an immigration background, including immigrants',
    'people with an immigration background': 'people with an immigration background, including immigrants',
    'racial and ethnic minorities': 'ethnic and racial minorities',
    'visible and ethnic minorities': 'ethnic and racial minorities',
    'ethnic minorities': 'ethnic and racial minorities',
    'minorities': 'ethnic and racial minorities',
    'east germans': 'ethnic and racial minorities',
    'west germans': 'ethnic and racial minorities',
    'ethnic germans': 'ethnic and racial minorities',
    'expatriates': 'ethnic and racial minorities',
    'white': 'ethnic and racial minorities',
    'white people': 'ethnic and racial minorities',
    'visible minorities': 'ethnic and racial minorities',
    'language and ethnic minorities': 'ethnic and racial minorities',
    'lgbtqi*': 'lgbtqia+', 'lgbtqqia+': 'lgbtqia+',
    'cis & heterosexuals': 'cisgender and heterosexuals',
    'religious groups': 'multiple (or other) religious or minority groups',
    'religious minorities': 'multiple (or other) religious or minority groups',
    'territorial language minorities': 'multiple (or other) religious or minority groups',
    'other minorities': 'multiple (or other) religious or minority groups',
    'disabled': 'disabled people',
    'other profession': 'other professions',
    'scientists': 'scientists and professors', 'professors': 'scientists and professors',
    'teachers': 'teachers and educators', 'educators': 'teachers and educators',
    'farmers': 'farmers and fishermen', 'fishermen': 'farmers and fishermen',
    'high-ranking officials': 'politicians and high-ranking officials',
    'politicians': 'politicians and high-ranking officials',
    'prostitutes': 'sex workers',
    'social professions': 'other professions',
    'engineers': 'other professions',
    'lobbyists': 'other professions',
    'hunters': 'other professions',
    'people working in the public sector': 'civil servants',
    'commuters': 'consumers and clients',
    'cyclists': 'car drivers', 'pedestrians': 'car drivers',
    'road carriers': 'consumers and clients',
    'air travellers': 'consumers and clients',
    'users of certain transportation modes': 'consumers and clients',
    'public transport passengers': 'consumers and clients',
    'insured persons': 'patients',
    'consumers': 'consumers and clients',
    'tax payers': 'others', 'gun owners': 'others',
    'tax evaders and white collar criminals': 'offenders, criminals, prisoners and/or accused people',
    'offenders': 'offenders, criminals, prisoners and/or accused people',
    'offenders or criminals': 'offenders, criminals, prisoners and/or accused people',
    'criminals': 'offenders, criminals, prisoners and/or accused people',
    'prisoners': 'offenders, criminals, prisoners and/or accused people',
    'terrorists': 'terrorists, rebels, revolutionaries and/or movements of armed resistance',
    'home owner': 'real-estate owners', 'land owner': 'real-estate owners',
    'landlords': 'real-estate owners', 'real-estate owner': 'real-estate owners',
    'real estate owners': 'real-estate owners',
    'inhabitants of cities': 'others',
    'inhabitants of rural or underserved areas': 'others',
    'inhabitants of other areas': 'others',
    'inhabitants of overseas': 'others',
    'inhabitants of specific sites': 'others',
    'inhabitants of underprivileged areas': 'others',
    'victims of crimes': 'others', 'victims of state violence': 'others',
    'victims of german history': 'others', 'whistle-blower and witnesses': 'others',
    'volunteers': 'others', 'people without public social protection': 'others',
    'heirs': 'others',
    'other': 'others',
    'target abroad': None,
}


def normalise_label(raw: str) -> str | None:
    raw = raw.strip().lower()
    if raw in ('target abroad', 'nan', ''):
        return None
    if raw in NORMALISE:
        return NORMALISE[raw]
    if raw in CANONICAL:
        return raw
    for canon in CANONICAL:
        if canon.lower() == raw:
            return canon
    return 'others'

## Label parsing

In [ ]:
def _protect(t: str) -> str:
    for cl in sorted(COMMA_LABELS, key=len, reverse=True):
        t = t.replace(cl, cl.replace(', ', '|||'))
    return t

def _unprotect(t: str) -> str:
    return t.replace('|||', ', ')


def parse_ann_label(val) -> frozenset:
    """Individual annotator label field (semicolon- and comma-separated strings)."""
    if pd.isna(val):
        return frozenset()
    val = str(val).strip()
    if val in ('nan', ''):
        return frozenset()
    protected = _protect(val.lower())
    labels = set()
    for chunk in re.split(r';\s*', protected):
        chunk = _unprotect(chunk).strip()
        if not chunk or chunk == 'nan':
            continue
        for piece in _protect(chunk).split(', '):
            piece = _unprotect(piece).strip()
            if piece:
                canon = normalise_label(piece)
                if canon is not None:
                    labels.add(canon)
    return frozenset(labels)


def parse_consensus_label(val) -> frozenset:
    """Consensus annotator span list [[start, end, 'label'], ...]."""
    if pd.isna(val):
        return frozenset()
    try:
        labels = set()
        for s in ast.literal_eval(str(val)):
            if isinstance(s, list) and len(s) >= 3:
                canon = normalise_label(str(s[2]))
                if canon is not None:
                    labels.add(canon)
        return frozenset(labels)
    except Exception:
        return frozenset()

## MASI distance and Krippendorff's α

In [ ]:
def masi_distance(A: frozenset, B: frozenset) -> float:
    if A == B:
        return 0.0
    inter = len(A & B)
    union = len(A | B)
    jaccard = inter / union if union > 0 else 0.0
    m = 0.67 if (A <= B or B <= A) else (0.33 if inter > 0 else 0.0)
    return 1.0 - jaccard * m


def alpha_masi(pairs) -> float:
    """Krippendorff's α with MASI distance on (annotator_labels, consensus_labels) pairs."""
    if not pairs:
        return np.nan
    D_o = sum(masi_distance(a, c) for a, c in pairs) / len(pairs)
    all_values = [v for pair in pairs for v in pair]
    n = len(all_values)
    if n < 2:
        return np.nan
    D_e = sum(
        masi_distance(all_values[i], all_values[j])
        for i in range(n) for j in range(n) if i != j
    ) / (n * (n - 1))
    return np.nan if D_e == 0 else round(1.0 - D_o / D_e, 4)

## Load and merge

In [ ]:
consensus_df = pd.read_excel(CONSENSUS_FILE)
ann_df = pd.read_excel(ANN_FILE)

consensus_df['outlet'] = consensus_df['outlet'].astype(str)
consensus_df = consensus_df[consensus_df['outlet'] != 'nan'].copy()

consensus_df['consensus_labels'] = consensus_df['ground_truth'].apply(parse_consensus_label)
ann_df['ann_labels'] = ann_df['label'].apply(parse_ann_label)

merged = ann_df.merge(
    consensus_df[['text', 'outlet', 'country', 'consensus_labels']],
    on='text', how='inner',
)
pairs_all = list(zip(merged['ann_labels'], merged['consensus_labels']))

## ICR by outlet

In [ ]:
outlet_rows = []
for outlet in sorted(merged['outlet'].dropna().unique()):
    sub = merged[merged['outlet'] == outlet]
    pairs = list(zip(sub['ann_labels'], sub['consensus_labels']))
    outlet_rows.append({
        'Outlet':                outlet,
        'Country':               sub['country'].iloc[0],
        'Krippendorff α (MASI)': alpha_masi(pairs),
        '# Sentences':           len(pairs),
        '# Annotators':          2,  # individual vs consensus, per sentence
    })

df_outlets = pd.DataFrame(outlet_rows)
df_outlets.to_csv(OUTPUT_DIR / 'icr_step2_by_outlet.csv', index=False)

print(f'{"Newspaper":<28} {"α (MASI)":>10}  {"N":>6}')
print('-' * 50)
for country_name, code in [('French Newspapers', 'France'), ('German Newspapers', 'Germany')]:
    print(country_name)
    sub = df_outlets[df_outlets['Country'] == code].sort_values('Outlet')
    for _, r in sub.iterrows():
        print(f'  {r["Outlet"]:<26} {r["Krippendorff α (MASI)"]:>10.2f}  {r["# Sentences"]:>6,}')
print('-' * 50)
print(f'  {"Overall":<26} {alpha_masi(pairs_all):>10.2f}  {len(pairs_all):>6,}')

df_outlets